# Un modelo de difusión 2D y un pipeline de diffusers

Notebook opcional del **módulo 13** del curso (en la app: `uv run dev` y la ruta `#/modulo/difusion`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

Primero entrenamos un pequeño predictor de ruido sobre datos 2D (espiral) con la pérdida DDPM.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

n = 4000
t = torch.rand(n) * 3 * math.pi + 0.5
x0 = torch.stack([0.28 * t * torch.cos(t), 0.28 * t * torch.sin(t)], 1) + 0.03 * torch.randn(n, 2)
T = 200
beta = torch.linspace(1e-4, 0.05, T)
ab = torch.cumprod(1 - beta, 0)
red = nn.Sequential(nn.Linear(3, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(red.parameters(), 1e-3)
for paso in range(3000):
    i = torch.randint(0, n, (256,))
    tt = torch.randint(0, T, (256,))
    eps = torch.randn(256, 2)
    xt = ab[tt, None].sqrt() * x0[i] + (1 - ab[tt, None]).sqrt() * eps
    perdida = ((red(torch.cat([xt, tt[:, None] / T], 1)) - eps) ** 2).mean()
    opt.zero_grad()
    perdida.backward()
    opt.step()
print("pérdida final:", perdida.item())

In [ ]:
x = torch.randn(1000, 2)
with torch.no_grad():
    for k in reversed(range(T)):
        e = red(torch.cat([x, torch.full((1000, 1), k / T)], 1))
        x = (x - beta[k] / (1 - ab[k]).sqrt() * e) / (1 - beta[k]).sqrt()
        if k > 0:
            x += beta[k].sqrt() * torch.randn_like(x)
plt.scatter(*x.T, s=2)
plt.title("Muestras generadas")
plt.show()

Un pipeline real de `diffusers` (con GPU es rápido; en CPU tarda): varía `guidance_scale` y `num_inference_steps`.

In [ ]:
from diffusers import AutoPipelineForText2Image

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo").to(dispositivo)
for g in [0.0, 1.5]:
    img = pipe("a watercolor painting of a transformer robot reading a book", num_inference_steps=2, guidance_scale=g).images[0]
    plt.imshow(img)
    plt.title(f"guidance_scale = {g}")
    plt.axis("off")
    plt.show()